In [0]:
df = spark.table("novacart_dev.bronze.exchange_rates")

In [0]:
from pyspark.sql import functions as F
df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

In [0]:
df = df.replace(["\\N", "?", "", "null", "NULL"], None)

In [0]:
string_cols = [c for c, t in df.dtypes if t == "string"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.col(col)))

df = df.withColumn("currency_code", F.upper(F.col("currency_code")))

In [0]:
df = df.dropDuplicates(["currency_code", "effective_date"])

In [0]:
df = df.withColumn(
    "effective_date",
    F.coalesce(
        F.to_date(F.expr("try_to_timestamp(effective_date, 'yyyy-MM-dd')")),
        F.to_date(F.expr("try_to_timestamp(effective_date, 'yyyy/MM/dd')")),
        F.to_date(F.expr("try_to_timestamp(effective_date, 'dd/MM/yyyy')"))
    )
)

In [0]:
df = df.withColumn(
    "exchange_rate_to_usd",
    F.col("exchange_rate_to_usd").cast("double")
)

df = df.withColumn(
    "exchange_rate_to_usd",
    F.when(F.col("exchange_rate_to_usd") <= 0, None)
     .otherwise(F.col("exchange_rate_to_usd"))
)

In [0]:
df = df.withColumn(
    "exchange_rate_to_usd",
    F.when(F.col("currency_code") == "USD", 1.0)
     .otherwise(F.col("exchange_rate_to_usd"))
)

In [0]:
df = df.withColumn(
    "dq_invalid_date",
    F.when(F.col("effective_date").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_currency",
    F.when(F.col("currency_code").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_exchange",
    F.when(F.col("exchange_rate_to_usd").isNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumn("load_timestamp", F.current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.silver.slv_exchange_rates")

In [0]:
display(spark.table("novacart_dev.silver.slv_exchange_rates"))